In [1]:
import os
import librosa
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout


In [2]:
# -------------------- SETTINGS -------------------- #

DATASET_PATH = "sounds"
SAMPLE_RATE = 22050
MEL_N_MELS = 128
DESIRED_LENGTH = 500  # Fixed time dimension for spectrograms

In [3]:
import os
import numpy as np

DATASET_PATH = "sounds"

classes = []

# read folder names
for folder in os.listdir(DATASET_PATH):
    if os.path.isdir(os.path.join(DATASET_PATH, folder)):
        classes.append(folder)

classes = sorted(classes)

np.save("label_classes.npy", classes)

print("✔ Saved label_classes.npy with:", classes)


✔ Saved label_classes.npy with: ['ambulance', 'firetruck', 'traffic']


In [4]:
# -------------------- AUDIO → MEL SPECTROGRAM -------------------- #
def audio_to_mel_spectrogram(file_path, n_mels=MEL_N_MELS):
    y, sr = librosa.load(file_path, sr=SAMPLE_RATE)
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=n_mels)
    mel_db = librosa.power_to_db(mel, ref=np.max)

    # Normalize size
    if mel_db.shape[1] < DESIRED_LENGTH:
        pad = DESIRED_LENGTH - mel_db.shape[1]
        mel_db = np.pad(mel_db, ((0, 0), (0, pad)), mode="constant")
    else:
        mel_db = mel_db[:, :DESIRED_LENGTH]

    return mel_db


In [5]:
# -------------------- DATASET PREPROCESSING -------------------- #
def preprocess_dataset(dataset_path):
    labels = []
    spectrograms = []

    if not os.path.exists(dataset_path):
        raise RuntimeError(f"❌ Dataset path not found: {dataset_path}")

    valid_extensions = (".mp3", ".wav", ".ogg")

    for folder in os.listdir(dataset_path):
        folder_path = os.path.join(dataset_path, folder)

        if os.path.isdir(folder_path):    
            audio_found = False

            for file in os.listdir(folder_path):
                if file.lower().endswith(valid_extensions):
                    audio_found = True
                    file_path = os.path.join(folder_path, file)
                    print(f"Processing: {file_path}")

                    try:
                        mel_spectrogram = audio_to_mel_spectrogram(file_path)
                        spectrograms.append(mel_spectrogram)
                        labels.append(folder)
                    except Exception as e:
                        print("Error:", file_path, "=>", e)

            if not audio_found:
                print(f"⚠ No audio files in {folder_path}")

    if len(labels) == 0:
        raise RuntimeError("❌ No valid dataset found! Check folder structure.")

    print("✔ Classes detected:", set(labels))
    print("✔ Total audio files loaded:", len(labels))

    return np.array(spectrograms), np.array(labels)


In [6]:
# -------------------- LOAD DATA -------------------- #
spectrograms, labels = preprocess_dataset(DATASET_PATH)

# Add channel dimension for CNN
spectrograms = np.expand_dims(spectrograms, axis=-1)

# Encode labels
label_encoder = LabelEncoder()
encoded_labels = label_encoder.fit_transform(labels)
one_hot_labels = to_categorical(encoded_labels)

# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    spectrograms, one_hot_labels, test_size=0.2, random_state=42
)

Processing: sounds\ambulance\sound_1.wav
Processing: sounds\ambulance\sound_10.wav
Processing: sounds\ambulance\sound_100.wav
Processing: sounds\ambulance\sound_101.wav
Processing: sounds\ambulance\sound_102.wav
Processing: sounds\ambulance\sound_103.wav
Processing: sounds\ambulance\sound_104.wav
Processing: sounds\ambulance\sound_105.wav
Processing: sounds\ambulance\sound_106.wav
Processing: sounds\ambulance\sound_107.wav
Processing: sounds\ambulance\sound_108.wav
Processing: sounds\ambulance\sound_109.wav
Processing: sounds\ambulance\sound_11.wav
Processing: sounds\ambulance\sound_110.wav
Processing: sounds\ambulance\sound_111.wav
Processing: sounds\ambulance\sound_112.wav
Processing: sounds\ambulance\sound_113.wav
Processing: sounds\ambulance\sound_114.wav
Processing: sounds\ambulance\sound_115.wav
Processing: sounds\ambulance\sound_116.wav
Processing: sounds\ambulance\sound_117.wav
Processing: sounds\ambulance\sound_118.wav
Processing: sounds\ambulance\sound_119.wav
Processing: sou

In [7]:
# -------------------- MODEL -------------------- #
model = Sequential([
    Conv2D(32, (3, 3), activation="relu", input_shape=(MEL_N_MELS, DESIRED_LENGTH, 1)),
    MaxPooling2D((2, 2)),
    Dropout(0.25),

    Conv2D(64, (3, 3), activation="relu"),
    MaxPooling2D((2, 2)),
    Dropout(0.25),

    Flatten(),
    Dense(128, activation="relu"),
    Dropout(0.5),
    Dense(len(label_encoder.classes_), activation="softmax")
])

model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])

print("\n✔ Training Started...\n")

history = model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=32,
    validation_split=0.2
)


C:\Users\user\anaconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)



✔ Training Started...

Epoch 1/20
12/12 ━━━━━━━━━━━━━━━━━━━━ 13s 917ms/step - accuracy: 0.3750 - loss: 59.8888 - val_accuracy: 0.6354 - val_loss: 1.0551
Epoch 2/20
12/12 ━━━━━━━━━━━━━━━━━━━━ 11s 910ms/step - accuracy: 0.5599 - loss: 1.2759 - val_accuracy: 0.8333 - val_loss: 0.9591
Epoch 3/20
12/12 ━━━━━━━━━━━━━━━━━━━━ 11s 922ms/step - accuracy: 0.7188 - loss: 0.7259 - val_accuracy: 0.8542 - val_loss: 0.7001
Epoch 4/20
12/12 ━━━━━━━━━━━━━━━━━━━━ 11s 908ms/step - accuracy: 0.8438 - loss: 0.4982 - val_accuracy: 0.8854 - val_loss: 0.3995
Epoch 5/20
12/12 ━━━━━━━━━━━━━━━━━━━━ 11s 902ms/step - accuracy: 0.9089 - loss: 0.2925 - val_accuracy: 0.9167 - val_loss: 0.2889
Epoch 6/20
12/12 ━━━━━━━━━━━━━━━━━━━━ 11s 882ms/step - accuracy: 0.9141 - loss: 0.2521 - val_accuracy: 0.9167 - val_loss: 0.3322
Epoch 7/20
12/12 ━━━━━━━━━━━━━━━━━━━━ 11s 883ms/step - accuracy: 0.9271 - loss: 0.1923 - val_accuracy: 0.8854 - val_loss: 0.2865
Epoch 8/20
12/12 ━━━━━━━━━━━━━━━━━━━━ 11s 886ms/step - accuracy: 0.9583 

In [8]:
# -------------------- EVALUATE -------------------- #
loss, accuracy = model.evaluate(X_test, y_test)
print(f"\n✔ Test Accuracy: {accuracy * 100:.2f}%")

# -------------------- SAVE MODEL -------------------- #
model.save("ambulance_sound.h5")
print("✔ Model Saved as ambulance_sound.h5")

4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 147ms/step - accuracy: 0.9583 - loss: 0.2646



✔ Test Accuracy: 95.83%
✔ Model Saved as bee_sound.h5


In [9]:
import tkinter as tk
from tkinter import filedialog, messagebox
import librosa
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import LabelEncoder
from playsound import playsound
import threading

# ==========================
# Model + settings
# ==========================

MODEL_FILE = "bee_sound.h5"
SAMPLE_RATE = 22050
MEL_N_MELS = 128
DESIRED_LENGTH = 500

# MATCH TRAINING CLASSES
classes = ['ambulance', 'firetruck', 'traffic']

encoder = LabelEncoder()
encoder.fit(classes)

# Load model
model = tf.keras.models.load_model(MODEL_FILE)

file_path = None

# ==========================
# Extract MEL features
# ==========================

def extract_mel(path):
    try:
        y, sr = librosa.load(path, sr=SAMPLE_RATE)
    except Exception:
        return None

    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=MEL_N_MELS)
    mel = librosa.power_to_db(mel, ref=np.max)

    if mel.shape[1] < DESIRED_LENGTH:
        pad = DESIRED_LENGTH - mel.shape[1]
        mel = np.pad(mel, ((0, 0), (0, pad)), mode="constant")
    else:
        mel = mel[:, :DESIRED_LENGTH]

    mel = np.expand_dims(np.expand_dims(mel, axis=-1), axis=0)
    return mel

# ==========================
# Play audio + Predict
# ==========================

def auto_process():
    global file_path

    # Play audio (threaded so UI won't freeze)
    def play_audio():
        try:
            playsound(file_path)
        except Exception as e:
            messagebox.showerror("Playback Error", str(e))

    threading.Thread(target=play_audio, daemon=True).start()

    # Predict audio
    mel = extract_mel(file_path)
    if mel is None:
        messagebox.showerror("Error", "Audio processing failed")
        return

    pred = model.predict(mel)
    idx = np.argmax(pred)
    result = encoder.inverse_transform([idx])[0]

    output_label.config(text=f"Prediction: {result}", fg="blue")

# ==========================
# Browse file → Auto Predict
# ==========================

def browse_and_predict():
    global file_path
    file_path = filedialog.askopenfilename(
        title="Select Audio File",
        filetypes=[("Audio Files", "*.wav *.mp3")]
    )

    if file_path:
        file_label.config(text=f"File: {file_path.split('/')[-1]}")
        root.after(300, auto_process)  # Delay for UI refresh

# ==========================
# UI Setup
# ==========================

root = tk.Tk()
root.title("Auto Audio Classifier")
root.geometry("450x220")

tk.Label(root, text="Select audio → auto play → auto classify", font=("Arial", 12)).pack(pady=10)

browse_btn = tk.Button(root, text="Select Audio File", width=25, command=browse_and_predict)
browse_btn.pack(pady=5)

file_label = tk.Label(root, text="No file selected", fg="gray")
file_label.pack(pady=5)

output_label = tk.Label(root, text="Prediction:", font=("Arial", 14))
output_label.pack(pady=10)

root.mainloop()


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 171ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 106ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 94ms/step


In [12]:
import tkinter as tk
from tkinter import messagebox
import numpy as np
import librosa
import tensorflow as tf
import sounddevice as sd
import noisereduce as nr
from sklearn.preprocessing import LabelEncoder
import threading

# ==========================
# Model + settings
# ==========================
MODEL_FILE = "bee_sound.h5"
SAMPLE_RATE = 22050
DURATION = 2.5               # seconds per prediction
MEL_N_MELS = 128
DESIRED_LENGTH = 500

classes = ['ambulance', 'firetruck', 'traffic']
encoder = LabelEncoder()
encoder.fit(classes)

model = tf.keras.models.load_model(MODEL_FILE)

is_listening = False

# ==========================
# Feature extraction
# ==========================
def extract_mel_from_audio(audio):
    mel = librosa.feature.melspectrogram(
        y=audio,
        sr=SAMPLE_RATE,
        n_mels=MEL_N_MELS
    )
    mel = librosa.power_to_db(mel, ref=np.max)

    if mel.shape[1] < DESIRED_LENGTH:
        mel = np.pad(mel, ((0, 0), (0, DESIRED_LENGTH - mel.shape[1])))
    else:
        mel = mel[:, :DESIRED_LENGTH]

    mel = mel.reshape(1, MEL_N_MELS, DESIRED_LENGTH, 1)
    return mel

# ==========================
# Realtime microphone loop
# ==========================
def listen_microphone():
    global is_listening

    while is_listening:
        try:
            # Record audio
            audio = sd.rec(
                int(DURATION * SAMPLE_RATE),
                samplerate=SAMPLE_RATE,
                channels=1,
                dtype='float32'
            )
            sd.wait()
            audio = audio.flatten()

            # Noise reduction
            reduced = nr.reduce_noise(y=audio, sr=SAMPLE_RATE)

            # Feature extraction
            mel = extract_mel_from_audio(reduced)

            # Prediction
            pred = model.predict(mel, verbose=0)
            idx = np.argmax(pred)
            label = encoder.inverse_transform([idx])[0]
            confidence = np.max(pred) * 100

            output_label.config(
                text=f"Prediction: {label} ({confidence:.1f}%)",
                fg="blue"
            )

        except Exception as e:
            messagebox.showerror("Error", str(e))
            break

# ==========================
# UI Controls
# ==========================
def start_listening():
    global is_listening
    if not is_listening:
        is_listening = True
        threading.Thread(target=listen_microphone, daemon=True).start()
        status_label.config(text="🎙 Listening...", fg="green")

def stop_listening():
    global is_listening
    is_listening = False
    status_label.config(text="⏹ Stopped", fg="red")

# ==========================
# UI Setup
# ==========================
root = tk.Tk()
root.title("Realtime Audio Classifier")
root.geometry("450x260")

tk.Label(root, text="Realtime Microphone Sound Recognition",
         font=("Arial", 13, "bold")).pack(pady=10)

start_btn = tk.Button(root, text="Start Microphone", width=20, command=start_listening)
start_btn.pack(pady=5)

stop_btn = tk.Button(root, text="Stop Microphone", width=20, command=stop_listening)
stop_btn.pack(pady=5)

output_label = tk.Label(root, text="Prediction:", font=("Arial", 14))
output_label.pack(pady=10)

status_label = tk.Label(root, text="⏹ Stopped", fg="red")
status_label.pack(pady=5)

root.mainloop()


In [1]:
plt.figure(figsize=(10,4))
plt.plot(history.history['accuracy'], label='Train Acc')
plt.plot(history.history['val_accuracy'], label='Val Acc')
plt.title("Accuracy Curve")
plt.xlabel("Epochs"); plt.ylabel("Accuracy")
plt.legend()
plt.show()

NameError: name 'plt' is not defined